Day 3: Exploratory Data Analysis (EDA).
Learning objectives
1. Understand the structure and quality of a dataset.
2. Calculate descriptive statistics.
3. Explore numerical and categorical variables.
4. Identify outliers and unusual patterns.
5. Formulate business questions from data.

In [104]:
import pandas as pd
import numpy as np

data = {
    "Product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Mouse",
                "Laptop", "Desk", "Keyboard", "Monitor", "Mouse"],
    "Category": ["Electronics", "Electronics", "Electronics", "Electronics",
                 "Electronics", "Electronics", "Furniture", "Electronics",
                 "Electronics", "Electronics"],
    "Region": ["North", "South", "North", "West", "South",
               "East", "East", "North", "West", "South"],
    "Quantity": [2, 5, 3, 1, 4, 1, 2, 6, 2, 8],
    "UnitPrice": [800, 20, 50, 250, 20, 800, 300, 50, 250, 20]
}

df = pd.DataFrame(data)
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [12]:
df.shape
df.head(5)
df.dtypes
df.isna().sum()

Product      0
Category     0
Region       0
Quantity     0
UnitPrice    0
Revenue      0
dtype: int64

In [105]:
df

,Product,Category,Region,Quantity,UnitPrice,Revenue
0,Laptop,Electronics,North,2,800,1600
1,Mouse,Electronics,South,5,20,100
2,Keyboard,Electronics,North,3,50,150
3,Monitor,Electronics,West,1,250,250
4,Mouse,Electronics,South,4,20,80
5,Laptop,Electronics,East,1,800,800
6,Desk,Furniture,East,2,300,600
7,Keyboard,Electronics,North,6,50,300
8,Monitor,Electronics,West,2,250,500
9,Mouse,Electronics,South,8,20,160


In [15]:
df.describe().round(2)

,Quantity,UnitPrice,Revenue
count,10.00,10.00,10.00
mean,3.40,256.00,454.00
std,2.32,306.57,467.36
min,1.00,20.00,80.00
25%,2.00,27.50,152.50
50%,2.50,150.00,275.00
75%,4.75,287.50,575.00
max,8.00,800.00,1600.00


Right-skewed distribution: the right tail is longer than the left; typically mean > median.

Heavy-tailed distribution: extreme values occur more frequently than they would under a light-tailed reference distribution, such as a normal distribution.

Long-tailed distribution: a relatively small number of observations extend far into the high-value range.

For our retail dataset, I'd initially describe the revenue distribution as potentially right-skewed

In [23]:
med=df["Revenue"].median()
min_=df["Revenue"].min()
max_=df["Revenue"].max()
df["Revenue"].sort_values(ascending=False)

0    1600
5     800
6     600
8     500
7     300
3     250
9     160
2     150
1     100
4      80
Name: Revenue, dtype: int64

## Outlier Detection: Interquartile Range (IQR) Method

The **Interquartile Range (IQR)** measures the spread of the middle 50% of a dataset.

### 1. Quartiles

- **Q1 (First quartile):** 25th percentile.
- **Q2 (Second quartile):** 50th percentile (median).
- **Q3 (Third quartile):** 75th percentile.

### 2. Calculate the IQR

The IQR is the difference between the third and first quartiles:

$$
IQR = Q3 - Q1
$$

### 3. Define the outlier boundaries

**Lower boundary:**

$$
Q1 - 1.5 \times IQR
$$

**Upper boundary:**

$$
Q3 + 1.5 \times IQR
$$

### 4. Identify potential outliers

An observation is flagged as a potential outlier if:

- Its value is below the lower boundary, or
- Its value is above the upper boundary.

### 5. Important considerations

- An outlier is not necessarily an error.
- Outliers may represent valid but unusual observations.
- The IQR method identifies potential outliers, not automatically incorrect data.
- Always investigate flagged observations before deciding whether to remove or modify them.

In [28]:
q1=df["Revenue"].quantile(0.25)
q3=df["Revenue"].quantile(0.75)
iqr=q3-q1
l_bound=q1-1.5*iqr
u_bound=q3+1.5*iqr
print({"Lower_bound":l_bound, "Upper_bound":u_bound})

{'Lower_bound': -481.25, 'Upper_bound': 1208.75}


In [33]:
df.loc[(df["Revenue"]< l_bound)|(df["Revenue"]> u_bound)]

,Product,Category,Region,Quantity,UnitPrice,Revenue
0,Laptop,Electronics,North,2,800,1600


<h1 style="color:blue;">Outlier detection function with Pandas filtering</h1>

In [108]:
def find_outliers(df):
    dfnum=df.select_dtypes(include="number").drop(columns=["ProductID","OrderID"],
                                       errors="ignore").copy()
    out=[]
    for col in dfnum:
        q1=dfnum[col].quantile(0.25)
        q3=dfnum[col].quantile(0.75)
        iqr=q3-q1
        l_bound=q1-1.5*iqr
        u_bound=q3+1.5*iqr
        outvalue=dfnum.loc[(dfnum[col]< l_bound)|(dfnum[col]> u_bound)]
        out.append({"Columns":col,
                    "Outlier_count":len(outvalue)})
    return pd.DataFrame(out)
    

In [109]:
out=find_outliers(df)
out  

,Columns,Outlier_count
0,Quantity,0
1,UnitPrice,2
2,Revenue,1


In [75]:
out=find_outliers(df)
results=[]
for col,outlier_df in out.items():
    results.append({"Colums":col,
                    "Outlier_count":len(outlier_df)})
results
summary=pd.DataFrame(results)
summary

,Colums,Outlier_count
0,Quantity,0
1,UnitPrice,2
2,Revenue,1


In [62]:
out=find_outliers(df)
results=
for col,outlier_df in out.items():
    print("Colums:",col)
    print("Number of outliers:",len(outlier_df))
    print("---")

Colums: Quantity
Number of outliers: 0
---
Colums: UnitPrice
Number of outliers: 2
---
Colums: Revenue
Number of outliers: 1
---


<h1 style="color:blue;">Outlier detection function with Python Loop</h1>

In [99]:
def find_outliers(df):
    dfnum=df.select_dtypes(include="number").drop(columns=["ProductID","OrderID"],
                                       errors="ignore").copy()
    out=[]
    for col in dfnum.columns:
        count=0
        q1=dfnum[col].quantile(0.25)
        q3=dfnum[col].quantile(0.75)
        iqr=q3-q1
        l_bound=q1-1.5*iqr
        u_bound=q3+1.5*iqr
        for value in dfnum[col]:
            if value< l_bound or value>u_bound:
                count+=1
        out.append({"Columns":col,
                    "Outlier_count":count})
    return pd.DataFrame(out)

find_outliers(df)

,Columns,Outlier_count
0,Quantity,0
1,UnitPrice,2
2,Revenue,1


In [117]:
df["Product"].value_counts()
df["Product"].unique()

array(['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Desk'], dtype=object)

In [118]:
df["Product"].value_counts()

Product
Mouse       3
Laptop      2
Keyboard    2
Monitor     2
Desk        1
Name: count, dtype: int64

In [119]:
df["Region"].value_counts()

Region
North    3
South    3
West     2
East     2
Name: count, dtype: int64

In [120]:
df

,Product,Category,Region,Quantity,UnitPrice,Revenue
0,Laptop,Electronics,North,2,800,1600
1,Mouse,Electronics,South,5,20,100
2,Keyboard,Electronics,North,3,50,150
3,Monitor,Electronics,West,1,250,250
4,Mouse,Electronics,South,4,20,80
5,Laptop,Electronics,East,1,800,800
6,Desk,Furniture,East,2,300,600
7,Keyboard,Electronics,North,6,50,300
8,Monitor,Electronics,West,2,250,500
9,Mouse,Electronics,South,8,20,160


In [127]:
df.groupby("Product")[["Quantity","Revenue"]].sum()

,Quantity,Revenue
Product,,
Desk,2,600
Keyboard,9,450
Laptop,3,2400
Monitor,3,750
Mouse,17,340


In [128]:
df.groupby("Region")["Revenue"].sum().sort_values(ascending=False)

Region
North    2050
East     1400
West      750
South     340
Name: Revenue, dtype: int64

In [129]:
df[["Quantity","Revenue"]].corr()

,Quantity,Revenue
Quantity,1.000000,-0.475277
Revenue,-0.475277,1.000000


In [130]:
df[["Quantity", "UnitPrice", "Revenue"]].corr().round(2)


,Quantity,UnitPrice,Revenue
Quantity,1.00,-0.65,-0.48
UnitPrice,-0.65,1.00,0.90
Revenue,-0.48,0.90,1.00


In [135]:
def find_outliers(df):
    dfnum=df.select_dtypes(include="number").drop(columns=["ProductID","OrderID"],
                                       errors="ignore").copy()
    out=[]
    outlier_indices=[]
    for col in dfnum.columns:
        count=0
        q1=dfnum[col].quantile(0.25)
        q3=dfnum[col].quantile(0.75)
        iqr=q3-q1
        l_bound=q1-1.5*iqr
        u_bound=q3+1.5*iqr
        for idx, value in dfnum[col].items():
            if value< l_bound or value>u_bound:
                count+=1
                outlier_indices.append(idx)
        out.append({"Columns":col,
                    "Outlier_count":count})
        
    outliers_df=df.loc[sorted(set(outlier_indices))]
    return pd.DataFrame(out), outliers_df

find_outliers(df)

(     Columns  Outlier_count
 0   Quantity              0
 1  UnitPrice              2
 2    Revenue              1,
   Product     Category Region  Quantity  UnitPrice  Revenue
 0  Laptop  Electronics  North         2        800     1600
 5  Laptop  Electronics   East         1        800      800)